## Glooko

## Case 1

A patient records his/her blood glucose readings in an app. These readings are stored as simple hash eg.

{“readings”: { “bg_value”: 150, ‘timestamp’: ‘2015-04-15T11:15:34’}, {“bg_value”: 90, ‘timestamp’: ‘2015-04-16T11:15:34’}, {bg_value”: 250, ‘timestamp’: ‘2015-04-17T11:15:34’}}

Your program should be able to parse such a hash and store those readings. For these readings based on predefined thresholds the program should report and store alerts. The thresholds are typically upper and lower limits for healthy blood glucose readings. Also include alerts for really low blood glucose levels.

These readings can be shared with a particular health care professional by the patient. For the health care professional show the alerts that are only for the patients they have access to.

Make appropriate assumptions and ensure that the program works. Some form of evidence for this working program would be helpful in assessment.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import json

### Simple case: only one patient and 4 readings

In [2]:
raw_readings = {'readings' : 
            [{'bg_value': 150, 'timestamp': '2015-04-15T11:15:34'},
               {'bg_value': 90, 'timestamp': '2015-04-16T11:15:34'},
               {'bg_value': 250, 'timestamp': '2015-04-17T11:15:34'},
               {'bg_value': 65, 'timestamp': '2015-04-18T11:15:34'}
            ] }
raw_readings

{'readings': [{'bg_value': 150, 'timestamp': '2015-04-15T11:15:34'},
  {'bg_value': 90, 'timestamp': '2015-04-16T11:15:34'},
  {'bg_value': 250, 'timestamp': '2015-04-17T11:15:34'},
  {'bg_value': 65, 'timestamp': '2015-04-18T11:15:34'}]}

In [3]:
df=pd.DataFrame(raw_readings['readings'])

#we key in a random Patient name => 'Ross'

df['Patient']='Ross'
df

,bg_value,timestamp,Patient
0,150,2015-04-15T11:15:34,Ross
1,90,2015-04-16T11:15:34,Ross
2,250,2015-04-17T11:15:34,Ross
3,65,2015-04-18T11:15:34,Ross


In [298]:
def alert(row, threshold_inf=90,threshold_sup=140,threshold_too_low=70):
    
    '''
    Calculates for each measurement the type of alert using below thresholds:
    # Superior Threshold: 140 mg/dL
    # Inferior threshold: 90 mg/dL
    # Too low glucose (hypoglycemie): below 70 mg/dL
    
    Example output for a glucose level of 145 mg/dL : (1, Clucose too high)
    '''
    
    if row < 70:
        return 1,'Glucose too low'
    elif row > 140:
        return 1,'Glucose too high'
    else:
        return 0,'OK'

In [322]:
#We define patient-doctor mapping
#Eg. Dr House is Ross's doctor

df_doctor_patient = pd.DataFrame({
                        'Doctor':['Dr. House'],
                        'Patient':['Ross']
})

In [323]:
def doctor_monitoring(df,doctor):
    
    #Trigger alerts for glucose levels
    df['alert'],df['alert_desc']=zip(*df['bg_value'].map(alert))
    
    
    #Find the doctor for each patient
    df_doctor = pd.merge(df,df_doctor_patient,how='left',on='Patient')

    
    if df['alert'].max() == 0:
               
        return print('there are no alerts to highlight, all patients are healthy')
    
    else:
        
        print('There are some alerts, please check below: ')
        
        return df_doctor[(df_doctor['Doctor']==doctor) & (df_doctor['alert']==1)]

In [324]:
doctor_monitoring(df,'Dr. House')

There are some alerts, please check below: 


,bg_value,timestamp,Patient,alert,alert_desc,Doctor
0,150,2015-04-15T11:15:34,Ross,1,Glucose too high,Dr. House
2,250,2015-04-17T11:15:34,Ross,1,Glucose too high,Dr. House
3,65,2015-04-18T11:15:34,Ross,1,Glucose too low,Dr. House


## Generic case: 2 or more patients with n periods (days) of readings

For illustration purposes, we input some patients and doctors below : 

In [4]:
patients = ['Ross','Chandler','Monica','John','Stephen','Emily','Peter','Holga','Gustaf','Helga']

doctors = ['Dr. House']*3+['Dr. Christina Yang']*3+['Dr. Bones']*4

df_doctor_patient = pd.DataFrame({
                        'Doctor':['Dr. House']*3+['Dr. Christina Yang']*3+['Dr. Bones']*4,
                        'Patient':['Ross','Chandler','Monica','John','Stephen','Emily','Peter','Holga','Gustaf','Helga']
})

df_doctor_patient

,Doctor,Patient
0,Dr. House,Ross
1,Dr. House,Chandler
2,Dr. House,Monica
3,Dr. Christina Yang,John
4,Dr. Christina Yang,Stephen
5,Dr. Christina Yang,Emily
6,Dr. Bones,Peter
7,Dr. Bones,Holga
8,Dr. Bones,Gustaf
9,Dr. Bones,Helga


#### The following function 'readings_generator' generates random measures  for a number of patients and periods

In [5]:
def readings_generator(number_patients = 5, number_periods=31):
    '''
    Example output for 4 periods and 2 patients
    raw_readings = [{'readings' : 
                    [
                        { 'bg_value': 150, 'timestamp': '2015-04-15T11:15:34'},
                        {'bg_value': 90, 'timestamp': '2015-04-16T11:15:34'},
                        {'bg_value': 250, 'timestamp': '2015-04-17T11:15:34'},
                        {'bg_value': 65, 'timestamp': '2015-04-18T11:15:34'}
                    ]
                }
                ,
                
               {'readings' : 
                    [
                        {'bg_value': 180, 'timestamp': '2015-04-15T11:15:34'},
                        {'bg_value': 95, 'timestamp': '2015-04-16T11:15:34'},
                        {'bg_value': 220, 'timestamp': '2015-04-17T11:15:34'},
                        {'bg_value': 69, 'timestamp': '2015-04-18T11:15:34'}
                    ]
               }
            ]
    '''
    
    raw_readings = []


    for patient in patients[:number_patients]:
        
        bg_values = np.round(np.random.normal(100,30,number_periods),0)
        dates = pd.date_range(end = datetime.today(), periods=number_periods).tolist()
        
        dict_readings={}
        dict_readings['readings'] = {'bg_value':bg_values,
                                    'timestamp':dates
                                    }

        raw_readings.append(dict_readings)
    
    return raw_readings

In [6]:
def convert2df(raw_readings):
    
    '''
    Converts raw readings to a DataFrame + add the doctor
    '''
    raw_readings = readings_generator()
    df_readings=pd.DataFrame()
    
    for i, person in enumerate(raw_readings):
        
        patient = df_doctor_patient['Patient'][i]
        df_readings = pd.concat([df_readings,
                        pd.DataFrame(person['readings'],
                                    index=[patient]*len(person['readings']['bg_value'])
                                    )
                                ])

    df_readings = df_readings.reset_index().rename(columns={'index':'Patient'})
#     df_readings['day']=df_readings['timestamp'].dt.day
#     df_readings['month']=df_readings['timestamp'].dt.month
#     df_readings['hour']=df_readings['timestamp'].dt.hour
#     df_readings['minutes']=df_readings['timestamp'].dt.minute
        
    return df_readings

In [7]:
df_readings = convert2df(raw_readings)
df_readings

,Patient,bg_value,timestamp
0,Ross,100.0,2021-10-03 15:35:00.191728
1,Ross,91.0,2021-10-04 15:35:00.191728
2,Ross,155.0,2021-10-05 15:35:00.191728
3,Ross,79.0,2021-10-06 15:35:00.191728
4,Ross,113.0,2021-10-07 15:35:00.191728
...,...,...,...
150,Stephen,113.0,2021-10-29 15:35:00.194719
151,Stephen,95.0,2021-10-30 15:35:00.194719
152,Stephen,114.0,2021-10-31 15:35:00.194719
153,Stephen,125.0,2021-11-01 15:35:00.194719


In [348]:
def alert(row, threshold_inf=90,threshold_sup=140,threshold_too_low=70):
    
    '''
    Calculates for each measurement the type of alert using below thresholds:
    # Superior Threshold: 140 mg/dL
    # Inferior threshold: 90 mg/dL
    # Too low glucose (hypoglycemie): below 70 mg/dL
    
    Example output for a glucose level of 145 mg/dL : (1, Clucose too high)
    '''
    
    if row < 70:
        return 1,'Glucose too low'
    elif row > 140:
        return 1,'Glucose too high'
    else:
        return 0,'OK'

In [387]:
def doctor_monitoring(df,doctor='Dr. House'):
    
    #Trigger alerts for glucose levels
    df['alert'],df['alert_desc']=zip(*df['bg_value'].map(alert))
    
    
    #Find the doctor for each patient
    df_doctor = pd.merge(df,df_doctor_patient,how='left',on='Patient')

    
    if df['alert'].max() == 0:
               
        return print('there are no alerts to highlight, all patients are healthy')
    
    else:
        
        print('''\n The following patients : {} are not healthy : \n
              Please find below their respective readings outside normal thresholds: \n
              '''.format(df_doctor[df_doctor['alert']==1]['Patient'].unique())
             )
        
        return df_doctor[(df_doctor['Doctor']==doctor) & (df_doctor['alert']==1)]

In [388]:
doctor_monitoring(df_readings,'Dr. House')


 The following patients : ['Ross' 'Chandler' 'Monica' 'John' 'Stephen'] are not healthy : 

              Please find below their respective readings outside normal thresholds: 

              


,Patient,bg_value,timestamp,alert,alert_desc,Doctor
0,Ross,29.0,2021-10-02 15:02:47.956554,1,Glucose too low,Dr. House
1,Ross,147.0,2021-10-03 15:02:47.956554,1,Glucose too high,Dr. House
3,Ross,141.0,2021-10-05 15:02:47.956554,1,Glucose too high,Dr. House
8,Ross,56.0,2021-10-10 15:02:47.956554,1,Glucose too low,Dr. House
14,Ross,41.0,2021-10-16 15:02:47.956554,1,Glucose too low,Dr. House
28,Ross,152.0,2021-10-30 15:02:47.956554,1,Glucose too high,Dr. House
31,Chandler,58.0,2021-10-02 15:02:47.957552,1,Glucose too low,Dr. House
47,Chandler,165.0,2021-10-18 15:02:47.957552,1,Glucose too high,Dr. House
52,Chandler,61.0,2021-10-23 15:02:47.957552,1,Glucose too low,Dr. House
57,Chandler,156.0,2021-10-28 15:02:47.957552,1,Glucose too high,Dr. House


## Problem 2: Exercise is the best medicine

A patient records his/her exercise using an app on their phone. The exercise readings are stored as a simple hash eg.

{“exercises”: { ‘timestamp’: ‘2015-04-15T11:15:34’, ‘duration’: 25}, { “duration”: 90, ‘timestamp’: ‘2015-04-19T11:15:34’}, { “duration”: 25, ‘timestamp’: ‘2015-04-16T11:15:34’}}

where duration is in minutes.

Your program should be able to identify patterns and report them by week/month/90 days

Report the following

Best exercise days
Best exercise streak - a streak is 3 consecutive days of exercise
Worst exercise days
Worst exercise break - a break is more than 3 consecutive days of missed exercise
Any other interesting pattern you could identify



In [26]:
import random
random.seed(0)

In [23]:
# Define number of periods

number_periods=365

In [27]:
def exercice_readings_gen(number_periods=number_of_periods):
    '''
    Example output for 4 periods and 2 patients

    '''
    
    values = []
    dates = pd.date_range(end = datetime.today(), periods=number_periods).tolist()

    for i in range(number_of_periods):
        
        duration= np.round(np.random.normal(20,5),0) #duration is a normal variable of mean = 20 and std = 5
        value =   random.choice([0,duration])  #we incorporate randomness for days when person doesn't run
        
        values.append(value)
        
        
        
    dict_ex_readings={}
    dict_ex_readings['exercices'] = {'duration':values,
                                    'timestamp':dates
                                    }
    
    return dict_ex_readings

In [28]:
raw_exercice_readings = exercice_readings_gen(365)

In [458]:
pd.DataFrame(raw_exercice_readings['exercices'])

,duration,timestamp
0,0.0,2020-11-02 16:10:52.788918
1,0.0,2020-11-03 16:10:52.788918
2,24.0,2020-11-04 16:10:52.788918
3,13.0,2020-11-05 16:10:52.788918
4,0.0,2020-11-06 16:10:52.788918
...,...,...
360,22.0,2021-10-28 16:10:52.788918
361,16.0,2021-10-29 16:10:52.788918
362,0.0,2021-10-30 16:10:52.788918
363,0.0,2021-10-31 16:10:52.788918


In [557]:
def helper_convert2df(raw_exercice_readings):
    
    '''
    Converts raw readings to a DataFrame + add the doctor
    '''
    raw_exercice_readings = exercice_readings_gen(number_periods=365)
    df_exercices = pd.DataFrame(raw_exercice_readings['exercices'])
    
    df_exercices['date_block']=list(range(1,number_of_periods+1))
    df_exercices['day']=df_exercices['timestamp'].dt.day
    df_exercices['week']=df_exercices['timestamp'].dt.isocalendar().week
    df_exercices['month']=df_exercices['timestamp'].dt.month
    df_exercices['quarter']=df_exercices['timestamp'].dt.quarter
    df_exercices['exerciced']=(df_exercices['duration']>0).astype('int')
    df_exercices.loc[df_exercices['exerciced']==0,'exerciced']=-1    #We assign -1 to day where person doesn't run, it will be users for streaks    
    
    return df_exercices.sort_values(by='timestamp')

In [563]:
df_ex_readings = helper_convert2df(raw_exercice_readings)

print('''
        The readings span from {} to {} for a total of {} days
        '''.format(df_ex_readings.timestamp.min()
                   ,df_ex_readings.timestamp.max(),
                   number_of_periods)
     )
df_ex_readings


        The readings span from 2020-11-02 17:31:31.988915 to 2021-11-01 17:31:31.988915 for a total of 365 days
        


,duration,timestamp,date_block,day,week,month,quarter,exerciced
0,19.0,2020-11-02 17:31:31.988915,1,2,45,11,4,1
1,0.0,2020-11-03 17:31:31.988915,2,3,45,11,4,-1
2,0.0,2020-11-04 17:31:31.988915,3,4,45,11,4,-1
3,25.0,2020-11-05 17:31:31.988915,4,5,45,11,4,1
4,0.0,2020-11-06 17:31:31.988915,5,6,45,11,4,-1
...,...,...,...,...,...,...,...,...
360,19.0,2021-10-28 17:31:31.988915,361,28,43,10,4,1
361,27.0,2021-10-29 17:31:31.988915,362,29,43,10,4,1
362,13.0,2021-10-30 17:31:31.988915,363,30,43,10,4,1
363,0.0,2021-10-31 17:31:31.988915,364,31,43,10,4,-1


In [564]:
def lag_generator(df,col,lags=[1,2]):
    
    '''
    Helper function to calculate the best/worst streak
    '''
    
    
    for lag in lags:
        
        shifted = df[['date_block',col]].copy()
        shifted.columns = ['date_block',col+'_lag_'+str(lag)]
        shifted['date_block'] += lag
        df = pd.merge(df,shifted,on='date_block',how='left').fillna(0)
    
    return df

In [565]:
df_ex_readings = lag_generator(df_ex_readings,'exerciced',lags=[1,2])
df_ex_readings.head(3)

,duration,timestamp,date_block,day,week,month,quarter,exerciced,exerciced_lag_1,exerciced_lag_2
0,19.0,2020-11-02 17:31:31.988915,1,2,45,11,4,1,0.0,0.0
1,0.0,2020-11-03 17:31:31.988915,2,3,45,11,4,-1,1.0,0.0
2,0.0,2020-11-04 17:31:31.988915,3,4,45,11,4,-1,-1.0,1.0


In [566]:
def streaks(df, col):
    '''
    Helper function to calculate streaks
    '''
    sign = np.sign(df[col])
    s = sign.groupby((sign!=sign.shift()).cumsum()).cumsum()
    return df.assign(pos_streak=s.where(s>0, 0.0), neg_streak=s.where(s<0, 0.0).abs())

In [567]:
df_ex_readings = streaks(df_ex_readings,'exerciced')
df_ex_readings

,duration,timestamp,date_block,day,week,month,quarter,exerciced,exerciced_lag_1,exerciced_lag_2,pos_streak,neg_streak
0,19.0,2020-11-02 17:31:31.988915,1,2,45,11,4,1,0.0,0.0,1.0,0.0
1,0.0,2020-11-03 17:31:31.988915,2,3,45,11,4,-1,1.0,0.0,0.0,1.0
2,0.0,2020-11-04 17:31:31.988915,3,4,45,11,4,-1,-1.0,1.0,0.0,2.0
3,25.0,2020-11-05 17:31:31.988915,4,5,45,11,4,1,-1.0,-1.0,1.0,0.0
4,0.0,2020-11-06 17:31:31.988915,5,6,45,11,4,-1,1.0,-1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
360,19.0,2021-10-28 17:31:31.988915,361,28,43,10,4,1,-1.0,1.0,1.0,0.0
361,27.0,2021-10-29 17:31:31.988915,362,29,43,10,4,1,1.0,-1.0,2.0,0.0
362,13.0,2021-10-30 17:31:31.988915,363,30,43,10,4,1,1.0,1.0,3.0,0.0
363,0.0,2021-10-31 17:31:31.988915,364,31,43,10,4,-1,1.0,1.0,0.0,1.0


In [572]:
df_ex_readings.sort_values(by='duration',ascending=False)[:5]

,duration,timestamp,date_block,day,week,month,quarter,exerciced,exerciced_lag_1,exerciced_lag_2,pos_streak,neg_streak
148,33.0,2021-03-30 17:31:31.988915,149,30,13,3,1,1,1.0,-1.0,2.0,0.0
152,30.0,2021-04-03 17:31:31.988915,153,3,13,4,2,1,1.0,-1.0,2.0,0.0
7,30.0,2020-11-09 17:31:31.988915,8,9,46,11,4,1,1.0,1.0,3.0,0.0
266,29.0,2021-07-26 17:31:31.988915,267,26,30,7,3,1,-1.0,1.0,1.0,0.0
27,29.0,2020-11-29 17:31:31.988915,28,29,48,11,4,1,-1.0,-1.0,1.0,0.0


In [610]:
def calculate_abs_patterns(df):
    
    
    print('''ABSOLUTE METRICS''')
    
    '''AVERAGE EXERCICE'''
    
    avg_duration = df.duration.mean()
    
    print('''
            The average duration per day in the full time-period is {}
            '''.format(avg_duration)
         )
    
    '''MAX DURATION OF EXERCICE'''
    
    max_duration = df.duration.max()
    
    print('''
            The maximum duration in the full time-period is {} in day {}
            '''.format(max_duration,df.loc[df.duration==max_duration,'timestamp'].values ))
    
    '''MAX EXERCICE STREAK'''
    
    max_pos_streak = df.pos_streak.max()
    
    print('''
            The maximum quantity of exerciced days in a row in the full time-period is {} in  days {}
            '''.format(max_pos_streak,df.loc[df.pos_streak==max_pos_streak,'timestamp'].values ))

    '''MAX WORST STREAK'''
    
    max_neg_streak = df.neg_streak.max()
    
    print('''
            The maximum quantity of days without exercices in a row in the full time-period is {} in  days {}
            '''.format(max_neg_streak,df.loc[df.neg_streak==max_neg_streak,'timestamp'].values))
    


In [612]:
calculate_abs_patterns(df_ex_readings)

ABSOLUTE METRICS

            The average duration per day in the full time-period is 9.953424657534246
            

            The maximum duration in the full time-period is 33.0 in day ['2021-03-30T17:31:31.988915000']
            

            The maximum quantity of exerciced days in a row in the full time-period is 7.0 in  days ['2021-03-19T17:31:31.988915000' '2021-08-17T17:31:31.988915000']
            

            The maximum quantity of days without exercices in a row in the full time-period is 8.0 in  days ['2021-05-13T17:31:31.988915000']
            


In [638]:
def calculate_period_patterns(df,period='quarter'):
    

    
    print('''By Period METRICS''')

    '''AVERAGE EXERCICE'''
    
    avg_duration = df.groupby(period).duration.mean()
    
    print('''
            The average duration per day per {} chosen is \n\n {}
            '''.format(period,avg_duration)
         )
    
    '''MAX DURATION OF EXERCICE'''
    
    max_duration = df.groupby(period).duration.max()
    
    print('''
            The maximum duration in each per {} chosen is \n\n {}: 
            '''.format(period, max_duration)
         )
    
    
    '''MAX EXERCICE STREAK'''
    
    max_pos_streak = df.groupby(period).pos_streak.max()
    
    print('''
            The maximum quantity of exerciced days in a row per {} is \n\n {}
            '''.format(period, max_pos_streak)
         )
    
    '''MAX WORST STREAK'''
    
    max_neg_streak = df.groupby(period).neg_streak.max()
    
    print('''
            The maximum quantity of days without exercices in a row per {} is \n\n {}
            '''.format(period, max_neg_streak)
         )
    
    


In [639]:
calculate_period_patterns(df_ex_readings)

By Period METRICS

            The average duration per day per quarter chosen is 

 quarter
1     9.466667
2     8.890110
3    10.369565
4    11.065217
Name: duration, dtype: float64
            

            The maximum duration in each per quarter chosen is 

 quarter
1    33.0
2    30.0
3    29.0
4    30.0
Name: duration, dtype: float64: 
            

            The maximum quantity of exerciced days in a row per quarter is 

 quarter
1    7.0
2    5.0
3    7.0
4    5.0
Name: pos_streak, dtype: float64
            

            The maximum quantity of days without exercices in a row per quarter is 

 quarter
1    7.0
2    8.0
3    4.0
4    4.0
Name: neg_streak, dtype: float64
            


In [8]:
from tkinter import *  
  
top = Tk()  
sb = Scrollbar(top)  
sb.pack(side = RIGHT, fill = Y)  
  
mylist = Listbox(top, yscrollcommand = sb.set )  
  
for line in range(30):  
    mylist.insert(END, "Number " + str(line))  
  
mylist.pack( side = LEFT )  
sb.config( command = mylist.yview )  
  
mainloop()  

In [6]:
!pip install tkscrolledframe

In [1]:
#!/usr/bin/env python3

from tkinter import *
from tkscrolledframe import ScrolledFrame

# Create a root window
root = Tk()

# Create a ScrolledFrame widget
sf = ScrolledFrame(root, width=640, height=480)
sf.pack(side="top", expand=1, fill="both")

# Bind the arrow keys and scroll wheel
sf.bind_arrow_keys(root)
sf.bind_scroll_wheel(root)

# Create a frame within the ScrolledFrame
inner_frame = sf.display_widget(Frame)

# Add a bunch of widgets to fill some space
num_rows = 16
num_cols = 16
for row in range(num_rows):
    for column in range(num_cols):
        w = Label(inner_frame,
                  width=15,
                  height=5,
                  borderwidth=2,
                  relief="groove",
                  anchor="center",
                  justify="center",
                  text=str(row * num_cols + column))

        w.grid(row=row,
               column=column,
               padx=4,
               pady=4)

# Start Tk's event loop
root.mainloop()

In [3]:
from tkscrolledframe import ScrolledFrame
import tkinter as tk

# Create a root window
root = tk.Tk()

frame_top = tk.Frame(root, width=400, height=250)
frame_top.pack(side="top", expand=1, fill="both")

# Create a ScrolledFrame widget
sf = ScrolledFrame(frame_top, width=380, height=240)
sf.pack(side="top", expand=1, fill="both")

# Bind the arrow keys and scroll wheel
sf.bind_arrow_keys(frame_top)
sf.bind_scroll_wheel(frame_top)

frame = sf.display_widget(tk.Frame)

l = tk.Label(frame, text="Test text 1\nTest text 2\nTest text 3\nTest text 4\nTest text 5\nTest text 6\nTest text 7\nTest text 8\nTest text 9", font="-size 20")
l.pack()

root.mainloop()

In [84]:
import tkinter as tk
from tkinter import ttk

df=df_readings

root = tk.Tk()
container = ttk.Frame(root)
canvas = tk.Canvas(container)
scrollbar = ttk.Scrollbar(container, orient="vertical", command=canvas.yview)
scrollable_frame = ttk.Frame(canvas)

scrollable_frame.bind(
    "<Configure>",
    lambda e: canvas.configure(
        scrollregion=canvas.bbox("all")
    )
)

canvas.create_window((0, 0), window=scrollable_frame, anchor="nw")

canvas.configure(yscrollcommand=scrollbar.set)

# for i in range(50):
#     ttk.Label(scrollable_frame, text="Sample scrolling label").pack()
    
for i in range(len(df['bg_value'])):
    ttk.Label(scrollable_frame,
              text=str(df['bg_value'][i])+
                       ' mg/dL in day '+
                       str(df['timestamp'][i])[:10]).pack()    

container.pack()
canvas.pack(side="left", fill="both", expand=True)
scrollbar.pack(side="right", fill="y")

# root.mainloop()

from tkinter import *

root = Tk()

labelframe = LabelFrame(root, text="This is a LabelFrame")
labelframe.pack(fill="both", expand="yes")
 
left = Label(labelframe, text="Inside the LabelFrame")
left.pack()
 
root.mainloop()

In [85]:

# Python Program to make a scrollable frame
# using Tkinter
  
from tkinter import *
  
class ScrollBar:
     
    # constructor
    def __init__(self):
         
        # create root window
        root = Tk()
  
        # create a horizontal scrollbar by
        # setting orient to horizontal
        h = Scrollbar(root, orient = 'horizontal')
  
        # attach Scrollbar to root window at
        # the bootom
        h.pack(side = BOTTOM, fill = X)
  
        # create a vertical scrollbar-no need
        # to write orient as it is by
        # default vertical
        v = Scrollbar(root)
  
        # attach Scrollbar to root window on
        # the side
        v.pack(side = RIGHT, fill = Y)
          
  
        # create a Text widget with 15 chars
        # width and 15 lines height
        # here xscrollcomannd is used to attach Text
        # widget to the horizontal scrollbar
        # here yscrollcomannd is used to attach Text
        # widget to the vertical scrollbar
        t = Text(root, width = 15, height = 15, wrap = NONE,
                 xscrollcommand = h.set,
                 yscrollcommand = v.set)
  
        # insert some text into the text widget
        for i in range(len(df['bg_value'])):
            t.insert(END,str(df['bg_value'][i])+
                       ' mg/dL in day '+
                       str(df['timestamp'][i])[:10]+ '\n')
            
        
#         for i in range(len(df['bg_value'])):
#     ttk.Label(scrollable_frame,
#               text=str(df['bg_value'][i])+
#                        ' mg/dL in day '+
#                        str(df['timestamp'][i])[:10])
#                        .pack()   
  
        # attach Text widget to root window at top
        t.pack(side=TOP, fill=X)
  
        # here command represents the method to
        # be executed xview is executed on
        # object 't' Here t may represent any
        # widget
        h.config(command=t.xview)
  
        # here command represents the method to
        # be executed yview is executed on
        # object 't' Here t may represent any
        # widget
        v.config(command=t.yview)
  
        # the root window handles the mouse
        # click event
        root.mainloop()
 
# create an object to Scrollbar class
s = ScrollBar()

In [48]:
from tkinter import *

root = Tk()

labelframe = LabelFrame(root, text="This is a LabelFrame")
labelframe.pack(fill="both", expand="yes")
 
left = Label(labelframe, text="Inside the LabelFrame")
left.pack()
 
root.mainloop()

In [63]:
import tkinter as tk

def populate(frame):
    '''Put in some fake data'''
#     for row in range(100):
#         tk.Label(frame, text="%s" % row, width=3, borderwidth="1", 
#                  relief="solid").grid(row=row, column=0)
#         t="this is the second column for row %s" %row
#         tk.Label(frame, text=t).grid(row=row, column=1)

    for row in range(len(df['bg_value'])):
        tk.Label(frame, text="%s" % row, width=3, borderwidth="1", 
                 relief="solid").grid(row=row, column=0)
        t=(str(df['bg_value'][row])+' mg/dL in day '+
        str(df['timestamp'][row])[:10]+ '\n')
        tk.Label(frame, text=t).grid(row=row, column=1)

#     for i in range(len(df['bg_value'])):
#         t.insert(END,str(df['bg_value'][i])+
#                    ' mg/dL in day '+
#                    str(df['timestamp'][i])[:10]+ '\n')
        
def onFrameConfigure(canvas):
    '''Reset the scroll region to encompass the inner frame'''
    canvas.configure(scrollregion=canvas.bbox("all"))

root = tk.Tk()
canvas = tk.Canvas(root, borderwidth=0, background="#ffffff")
frame = tk.Frame(canvas, background="#ffffff")
vsb = tk.Scrollbar(root, orient="vertical", command=canvas.yview)
canvas.configure(yscrollcommand=vsb.set)

vsb.pack(side="right", fill="y")
canvas.pack(side="left", fill="both", expand=True)
canvas.create_window((4,4), window=frame, anchor="nw")

frame.bind("<Configure>", lambda event, canvas=canvas: onFrameConfigure(canvas))

populate(frame)

####
labelframe = LabelFrame(root, text="LabelFrame")
labelframe.pack(fill="both", expand="yes")
 
left = Label(labelframe, text="Inside the LabelFrame")
left.pack()

footer = ttk.Frame(root)
footer.pack()

root.mainloop()

### Another example

In [58]:
import tkinter as tk
from tkinter import ttk

class Scrollable(tk.Frame):
    """
       Make a frame scrollable with scrollbar on the right.
       After adding or removing widgets to the scrollable frame,
       call the update() method to refresh the scrollable area.
    """

    def __init__(self, frame, width=16):

        scrollbar = tk.Scrollbar(frame, width=width)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y, expand=False)

        self.canvas = tk.Canvas(frame, yscrollcommand=scrollbar.set)
        self.canvas.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

        scrollbar.config(command=self.canvas.yview)

        self.canvas.bind('<Configure>', self.__fill_canvas)

        # base class initialization
        tk.Frame.__init__(self, frame)

        # assign this obj (the inner frame) to the windows item of the canvas
        self.windows_item = self.canvas.create_window(0,0, window=self, anchor=tk.NW)


    def __fill_canvas(self, event):
        "Enlarge the windows item to the canvas width"

        canvas_width = event.width
        self.canvas.itemconfig(self.windows_item, width = canvas_width)

    def update(self):
        "Update the canvas and the scrollregion"

        self.update_idletasks()

In [59]:
root = tk.Tk()

header = ttk.Frame(root)
body = ttk.Frame(root)
footer = ttk.Frame(root)
header.pack()
body.pack()
footer.pack()

ttk.Label(header, text="The header").pack()
ttk.Label(footer, text="The Footer").pack()


scrollable_body = Scrollable(body, width=32)

for i in range(30):
    ttk.Button(scrollable_body, text="I'm a button in the scrollable frame").grid()

scrollable_body.update()

root.mainloop()

## Another example

In [66]:
from tkinter import *   # from x import * is bad practice
# from ttk import *

# http://tkinter.unpythonic.net/wiki/VerticalScrolledFrame

class VerticalScrolledFrame(Frame):
    """A pure Tkinter scrollable frame that actually works!
    * Use the 'interior' attribute to place widgets inside the scrollable frame
    * Construct and pack/place/grid normally
    * This frame only allows vertical scrolling

    """
    def __init__(self, parent, *args, **kw):
        Frame.__init__(self, parent, *args, **kw)            

        # create a canvas object and a vertical scrollbar for scrolling it
        vscrollbar = Scrollbar(self, orient=VERTICAL)
        vscrollbar.pack(fill=Y, side=RIGHT, expand=FALSE)
        canvas = Canvas(self, bd=0, highlightthickness=0,
                        yscrollcommand=vscrollbar.set)
        canvas.pack(side=LEFT, fill=BOTH, expand=TRUE)
        vscrollbar.config(command=canvas.yview)

        # reset the view
        canvas.xview_moveto(0)
        canvas.yview_moveto(0)

        # create a frame inside the canvas which will be scrolled with it
        self.interior = interior = Frame(canvas)
        interior_id = canvas.create_window(0, 0, window=interior,
                                           anchor=NW)

        # track changes to the canvas and frame width and sync them,
        # also updating the scrollbar
        def _configure_interior(event):
            # update the scrollbars to match the size of the inner frame
            size = (interior.winfo_reqwidth(), interior.winfo_reqheight())
            canvas.config(scrollregion="0 0 %s %s" % size)
            if interior.winfo_reqwidth() != canvas.winfo_width():
                # update the canvas's width to fit the inner frame
                canvas.config(width=interior.winfo_reqwidth())
        interior.bind('<Configure>', _configure_interior)

        def _configure_canvas(event):
            if interior.winfo_reqwidth() != canvas.winfo_width():
                # update the inner frame's width to fill the canvas
                canvas.itemconfigure(interior_id, width=canvas.winfo_width())
        canvas.bind('<Configure>', _configure_canvas)


if __name__ == "__main__":

    class SampleApp(Tk):
        def __init__(self, *args, **kwargs):
            root = Tk.__init__(self, *args, **kwargs)


            self.frame = VerticalScrolledFrame(root)
            self.frame.pack()
            self.label = Label(text="Shrink the window to activate the scrollbar.")
            self.label.pack()
            buttons = []
            for i in range(10):
                buttons.append(Button(self.frame.interior, text="Button " + str(i)))
                buttons[-1].pack()

    app = SampleApp()
    app.mainloop()

In [67]:
class ScrolledWindow(tk.Frame):
    """
    1. Master widget gets scrollbars and a canvas. Scrollbars are connected 
    to canvas scrollregion.

    2. self.scrollwindow is created and inserted into canvas

    Usage Guideline:
    Assign any widgets as children of <ScrolledWindow instance>.scrollwindow
    to get them inserted into canvas

    __init__(self, parent, canv_w = 400, canv_h = 400, *args, **kwargs)
    docstring:
    Parent = master of scrolled window
    canv_w - width of canvas
    canv_h - height of canvas

    """


    def __init__(self, parent, canv_w = 400, canv_h = 400, *args, **kwargs):
        """Parent = master of scrolled window
        canv_w - width of canvas
        canv_h - height of canvas

       """
        super().__init__(parent, *args, **kwargs)

        self.parent = parent

        # creating a scrollbars
        self.xscrlbr = ttk.Scrollbar(self.parent, orient = 'horizontal')
        self.xscrlbr.grid(column = 0, row = 1, sticky = 'ew', columnspan = 2)         
        self.yscrlbr = ttk.Scrollbar(self.parent)
        self.yscrlbr.grid(column = 1, row = 0, sticky = 'ns')         
        # creating a canvas
        self.canv = tk.Canvas(self.parent)
        self.canv.config(relief = 'flat',
                         width = 10,
                         heigh = 10, bd = 2)
        # placing a canvas into frame
        self.canv.grid(column = 0, row = 0, sticky = 'nsew')
        # accociating scrollbar comands to canvas scroling
        self.xscrlbr.config(command = self.canv.xview)
        self.yscrlbr.config(command = self.canv.yview)

        # creating a frame to inserto to canvas
        self.scrollwindow = ttk.Frame(self.parent)

        self.canv.create_window(0, 0, window = self.scrollwindow, anchor = 'nw')

        self.canv.config(xscrollcommand = self.xscrlbr.set,
                         yscrollcommand = self.yscrlbr.set,
                         scrollregion = (0, 0, 100, 100))

        self.yscrlbr.lift(self.scrollwindow)        
        self.xscrlbr.lift(self.scrollwindow)
        self.scrollwindow.bind('<Configure>', self._configure_window)  
        self.scrollwindow.bind('<Enter>', self._bound_to_mousewheel)
        self.scrollwindow.bind('<Leave>', self._unbound_to_mousewheel)

        return

    def _bound_to_mousewheel(self, event):
        self.canv.bind_all("<MouseWheel>", self._on_mousewheel)   

    def _unbound_to_mousewheel(self, event):
        self.canv.unbind_all("<MouseWheel>") 

    def _on_mousewheel(self, event):
        self.canv.yview_scroll(int(-1*(event.delta/120)), "units")  

    def _configure_window(self, event):
        # update the scrollbars to match the size of the inner frame
        size = (self.scrollwindow.winfo_reqwidth(), self.scrollwindow.winfo_reqheight())
        self.canv.config(scrollregion='0 0 %s %s' % size)
        if self.scrollwindow.winfo_reqwidth() != self.canv.winfo_width():
            # update the canvas's width to fit the inner frame
            self.canv.config(width = self.scrollwindow.winfo_reqwidth())
        if self.scrollwindow.winfo_reqheight() != self.canv.winfo_height():
            # update the canvas's width to fit the inner frame
            self.canv.config(height = self.scrollwindow.winfo_reqheight())

In [86]:
from tkinter import (Tk, Frame, Label, Scrollbar, Canvas,
                     DoubleVar, Entry, Button, StringVar, TclError)
from tkinter.ttk import Progressbar


class EndlessScroll(Canvas):
    def __init__(self, master, update_mode='passive', scrollbar=None, **kwargs):
        """update_mode:
                        'passive' (recommended and default): widgets have to be accessed (Text, Entry...)
                                  or using Frame, DON'T use if only single widget per line,
                                  USE Frame with this
                        'active': recommended only if single widget
                                  per line (best with Label
                                            but can use Button too)"""
        Canvas.__init__(self, master, **kwargs)
        self.master = master
        self._widget_heights = []
        self._start_index = 0
        self._widgets = []
        self.temp_frame = None
        self.__got_height = False
        self.__initialised_placement = False
        self.__height_counter = 0
        self.yscrollincrement = None
        self._update_mode = update_mode
        self.scrollbar = scrollbar
        self.__load_progress_tracker = DoubleVar(master=self.master, value=0.0)
        self.__percent_tracker = StringVar(master=self.master, value='0.0%')
        self._allow_mouse_control = False
        self.bind('<Enter>', lambda e: setattr(self, '_allow_mouse_control', True))
        self.bind('<Leave>', lambda e: setattr(self, '_allow_mouse_control', False))
        self.bind('<MouseWheel>', self.__scroll_with_mouse)
        for key, value in kwargs.items():
            setattr(self, key, value)

    def __scroll_with_mouse(self, event):
        if not self._allow_mouse_control:
            return
        self.yscroll('scroll', (-1 * (event.delta/120)), 'units')

    def set_scrollbar(self, scrollbar):
        self.scrollbar = scrollbar

    def init_after_widgets(self):
        self.temp_frame = Frame(self)
        self._widgets = self.winfo_children()[:-1]
        self.temp_frame.place(x=0, y=0, relwidth=1, relheight=1)
        self.update()
        Label(self.temp_frame, text='Loading...').pack(expand=True)
        Progressbar(self.temp_frame,
                    orient='horizontal',
                    variable=self.__load_progress_tracker,
                    length=self.temp_frame.winfo_width()
                           - self.temp_frame.winfo_width() // 10).pack(expand=True)
        Label(self.temp_frame, textvariable=self.__percent_tracker).pack(expand=True)
        self.after(100, self._get_widget_heights, 0)

    def _get_widget_heights(self, index):
        length = len(self._widgets)
        self.after(100, self.__get_widget_heights, index, self._widgets, length)

    def __get_widget_heights(self, index, widgets, length):
        if index >= length:
            self.__got_height = True
            self.temp_frame.place_forget()
            self._initial_placement(widgets)
            self.update_idletasks()
            self.__update()
            self.yscroll(None, 0)
            return
        id_ = self.create_window(0, 0, window=widgets[index], anchor='nw')
        self.update()
        try:
            self._widget_heights.append(widgets[index].winfo_height())
            self.delete(id_)
        except TclError:
            pass
        percent = (index + 1) * 100 / length
        self.__load_progress_tracker.set(value=percent)
        self.__percent_tracker.set(value=f'{percent: .2f}%')
        self.after(1, self.__get_widget_heights, index + 1, widgets, length)

    def yscroll(self, *args):
        if not self.__initialised_placement or not self.scrollbar:
            return
        type_, fraction = args[0], args[1]
        parent_height = self.winfo_height()
        if type_ == 'scroll':
            units = args[2]
            k = float(fraction)
            if k < -1.0:
                k = -1.0
            elif k > 1.0:
                k = 1.0
            top, bottom, *_ = self.scrollbar.get()
            length = (bottom - top) if not self.yscrollincrement else self.yscrollincrement
            if (top == 0.0 and k < 0) or (bottom == 1.0 and k > 0):
                return
            if 0 < top < length:
                length = top
            if 0 < (1 - bottom) < length:
                length = 1 - bottom
            if units == 'units':
                fraction = top + (k * length)
            elif units == 'pages':
                return
                # fraction = top + (k * (0.9 * parent_height) * length)
        sum_height = sum(self._widget_heights)
        scroll_height = float(fraction) * sum_height
        height_counter = 0
        for index_, height in enumerate(self._widget_heights):
            height_counter += height
            if height_counter > scroll_height:
                self._start_index = index_
                self.__height_counter = -(height - (height_counter - scroll_height))
                break
        self.scrollbar.set(float(fraction), float(fraction) + parent_height / sum_height)
        if self._update_mode == 'passive':
            self.__update()

    def __update(self, _call_from_self=False):
        if _call_from_self and self._update_mode == 'passive':
            self.after(10, self.__update, True)
            return
        parent_height = self.winfo_height()
        height_counter = self.__height_counter
        self.delete('all')
        for height, widget in zip(self._widget_heights[self._start_index:], self._widgets[self._start_index:]):
            if height_counter > parent_height:
                break
            self.create_window(0, height_counter, window=widget, anchor='nw')
            height_counter += height
        self.after(10, self.__update, True)

    def _initial_placement(self, widgets):
        parent_height = self.winfo_height()
        height_counter = 0
        for height, widget in zip(self._widget_heights, widgets):
            if height_counter > parent_height:
                self.__initialised_placement = True
                return
            self.create_window(0, height_counter, window=widget, anchor='nw')
            height_counter += height


def create_scroller():
    endless_scroll = EndlessScroll(root, yscrollincrement=0.1)
    endless_scroll.pack(side='left')

    for i in range(100):
        frame = Frame(endless_scroll)
        Label(frame, text=str(i).zfill(4)).pack()

    scrollbar = Scrollbar(root, command=endless_scroll.yscroll)
    scrollbar.pack(side='right', fill='y')

    endless_scroll.set_scrollbar(scrollbar)
    endless_scroll.init_after_widgets()


root = Tk()

create_scroller()

root.mainloop()

In [87]:
from tkinter import Tk, Canvas, Frame, Label, Scrollbar, Button, DoubleVar, StringVar, Entry
from tkinter.ttk import Progressbar


class PagedScrollFrame(Frame):
    def __init__(self, master, items_per_page=100, **kwargs):
        Frame.__init__(self, master, **kwargs)
        self.master = master
        self.items_per_page = items_per_page
        self.pages = None
        self.id_list = []
        self.bbox_tag = 'all'

        self._loading_frame = Frame(self)
        self.__load_progress_tracker = DoubleVar(master=self.master, value=0.0)
        self.__percent_tracker = StringVar(master=self.master, value='0.00%')

        self.frame = Frame(self)
        self.frame.pack(side='top', padx=20, pady=20)

        self.canvas = Canvas(self.frame)
        self.canvas.pack(side='left')

        self.bg_label = Label(self.canvas)
        self.bg_label.place(x=0, y=0, relwidth=1, relheight=1)

        self.scrollbar = Scrollbar(self.frame, orient='vertical', command=self.canvas.yview)
        self.scrollbar.pack(side='right', fill='y')
        self.canvas.config(yscrollcommand=self.scrollbar.set)
        self.canvas.bind('<Configure>',
                         lambda e: self.canvas.config(
                             scrollregion=self.canvas.bbox(self.bbox_tag)
                         ))

        self.button_frame = Frame(self)
        self.button_frame.pack(fill='x', side='bottom', padx=20, pady=20)

        self.canvas_frame = Frame(self.button_frame)
        self.button_canvas = Canvas(self.canvas_frame, height=20)
        self.button_canvas.pack(expand=True)
        self.inner_frame = Frame(self.button_canvas)
        self.button_canvas.create_window(0, 0, window=self.inner_frame, anchor='nw')

        self.button_scrollbar = Scrollbar(self.canvas_frame,
                                          orient='horizontal',
                                          command=self.button_canvas.xview)
        self.button_scrollbar.pack(fill='x')
        self.button_canvas.config(xscrollcommand=self.button_scrollbar.set)
        self.button_canvas.bind(
            '<Configure>', lambda e: self.button_canvas.config(
                scrollregion=self.button_canvas.bbox('all')
            )
        )

    def pack_items(self):
        if not self.pages:
            return
        self._loading_frame.place(x=0, y=0, relwidth=1, relheight=1)
        self._loading_frame.lift()
        self._loading_frame.update_idletasks()
        self.after(100, self._pack_items)

    def _pack_items(self):
        Label(self._loading_frame, text='Loading...').pack(expand=True)
        Progressbar(self._loading_frame,
                    orient='horizontal',
                    variable=self.__load_progress_tracker,
                    length=self._loading_frame.winfo_width()
                           - self._loading_frame.winfo_width() // 10).pack(expand=True)
        Label(self._loading_frame, textvariable=self.__percent_tracker).pack(expand=True)
        self.update_idletasks()
        widgets = [widget for page in self.pages for widget in page.winfo_children()]
        length = len(widgets)
        self.after(100, self.__pack_items, widgets, 0, length)

    def __pack_items(self, widgets, index, length):
        if index >= length:
            self._loading_frame.destroy()
            self.canvas.config(scrollregion=self.canvas.bbox('all'))
            return
        widgets[index].pack()
        percent = (index + 1) * 100 / length
        self.__load_progress_tracker.set(value=percent)
        self.__percent_tracker.set(value=f'{percent: .2f}%')
        self.after(1, self.__pack_items, widgets, index + 1, length)

    def change_frame(self, index):
        if not self.pages:
            return
        self.bbox_tag = self.id_list[index]
        self.canvas.config(scrollregion=self.canvas.bbox(self.bbox_tag))
        self.bg_label.lift()
        self.pages[index].lift()

    def create_pages(self, num_of_items, items_per_page=None):
        self.pages = None
        if not items_per_page:
            items_per_page = self.items_per_page
        num_of_pages = num_of_items // items_per_page
        if num_of_items % items_per_page != 0:
            num_of_pages += 1
        start_indexes = [items_per_page * page_num for page_num in range(num_of_pages)]
        end_indexes = [num + items_per_page for num in start_indexes]
        end_indexes[-1] += (num_of_items % items_per_page
                            - (items_per_page if num_of_items % items_per_page != 0 else 0))
        self.pages = [Frame(self.canvas) for _ in range(num_of_pages)]
        self.id_list = []
        for page, frame in enumerate(self.pages):
            self.id_list.append(self.canvas.create_window(0, 0, window=frame, anchor='nw'))
        self.pages[0].lift()
        if num_of_pages >= 2:
            Button(self.button_frame, text='1',
                   command=lambda: self.change_frame(0)).pack(
                side='left', expand=True, fill='both', ipadx=5
            )
            if num_of_pages > 2:
                self.canvas_frame.pack(fill='x', expand=True, side='left')
                for page_num in range(1, num_of_pages - 1):
                    Button(self.inner_frame, text=page_num + 1,
                           command=lambda index=page_num: self.change_frame(index)).pack(
                        expand=True, fill='both', side='left', ipadx=5
                    )
            Button(self.button_frame, text=num_of_pages,
                   command=lambda: self.change_frame(num_of_pages - 1)).pack(
                side='right', fill='both', expand=True, ipadx=5
            )
        return zip(start_indexes, end_indexes, self.pages)


def create_paged_canvas():
    scroll = PagedScrollFrame(root)
    scroll.pack()

    lst = tuple(range(3000))
    for start, end, parent in scroll.create_pages(len(lst)):
        for i in lst[start:end]:
            frame_ = Frame(parent)
            Label(frame_, text=str(i).zfill(4)).pack(side='left')

    scroll.pack_items()


root = Tk()
root.protocol('WM_DELETE_WINDOW', exit)

create_paged_canvas()

root.mainloop()

KeyboardInterrupt: 

In [13]:
import tkinter as tk

root = tk.Tk()
root.grid_rowconfigure(0, weight=1)
root.columnconfigure(0, weight=1)

frame_main = tk.Frame(root, bg="gray")
frame_main.grid(sticky='news')

label1 = tk.Label(frame_main, text="Label 1", fg="green")
label1.grid(row=0, column=0, pady=(5, 0), sticky='nw')

label2 = tk.Label(frame_main, text="Label 2", fg="blue")
label2.grid(row=1, column=0, pady=(5, 0), sticky='nw')

label3 = tk.Label(frame_main, text="Label 3", fg="red")
label3.grid(row=3, column=0, pady=5, sticky='nw')

# Create a frame for the canvas with non-zero row&column weights
frame_canvas = tk.Frame(frame_main)
frame_canvas.grid(row=2, column=0, pady=(5, 0), sticky='nw')
frame_canvas.grid_rowconfigure(0, weight=1)
frame_canvas.grid_columnconfigure(0, weight=1)
# Set grid_propagate to False to allow 5-by-5 buttons resizing later
frame_canvas.grid_propagate(False)

# Add a canvas in that frame
canvas = tk.Canvas(frame_canvas, bg="yellow")
canvas.grid(row=0, column=0, sticky="news")

# Link a scrollbar to the canvas
vsb = tk.Scrollbar(frame_canvas, orient="vertical", command=canvas.yview)
vsb.grid(row=0, column=1, sticky='ns')
canvas.configure(yscrollcommand=vsb.set)

# Create a frame to contain the buttons
frame_buttons = tk.Frame(canvas, bg="blue")
canvas.create_window((0, 0), window=frame_buttons, anchor='nw')

# Add 9-by-5 buttons to the frame
rows = 9
columns = 5
buttons = [[tk.Button() for j in range(columns)] for i in range(rows)]
for i in range(0, rows):
    for j in range(0, columns):
        buttons[i][j] = tk.Button(frame_buttons, text=("%d,%d" % (i+1, j+1)))
        buttons[i][j].grid(row=i, column=j, sticky='news')

# Update buttons frames idle tasks to let tkinter calculate buttons sizes
frame_buttons.update_idletasks()

# Resize the canvas frame to show exactly 5-by-5 buttons and the scrollbar
first5columns_width = sum([buttons[0][j].winfo_width() for j in range(0, 5)])
first5rows_height = sum([buttons[i][0].winfo_height() for i in range(0, 5)])
frame_canvas.config(width=first5columns_width + vsb.winfo_width(),
                    height=first5rows_height)

# Set the canvas scrolling region
canvas.config(scrollregion=canvas.bbox("all"))

# Launch the GUI
root.mainloop()